# Project: 2DEG mobility calculation using mobilitypy¶

The mobility models are implemented using the following refs (and refs therein):

Ref-1: J. Bassaler, J. Mehta, I. Abid, L. Konczewicz, S. Juillaguet, S. Contreras, S. Rennesson, S. Tamariz, M. Nemoz, F. Semond, J. Pernot, F. Medjdoub, Y. Cordier, P. Ferrandis, Al-Rich AlGaN Channel High Electron Mobility Transistors on Silicon: A Relevant Approach for High Temperature Stability of Electron Mobility. Adv. Electron. Mater. 2024, 2400069. https://doi.org/10.1002/aelm.202400069

Ref-2: Zhang, J., Hao, Y., Zhang, J. et al. The mobility of two-dimensional electron gas in AlGaN/GaN heterostructures with varied Al content. Sci. China Ser. F-Inf. Sci. 51, 780–789 (2008). https://doi.org/10.1007/s11432-008-0056-7

Ref-3:  B. Mondal, P. Pampili, J. Mukherjee, D. Moran, P.J. Parbrook, S. Schulz, Interplay of carrier density and mobility in Al-rich (Al,Ga)N-channel HEMTs: Impact on high-power device performance potential. APL Electronic Devices 1, 026117 (2025). https://doi.org/10.1063/5.0277051

# 1. Settings

## 1.1 Import modules

In [1]:
import sys, os
from pathlib import Path
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
import pandas as pd
import mobilitypy 
from mobilitypy import AlloyParams, Mobility2DCarrier, Plottings

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
print(f'mobilitypy version:{mobilitypy.__version__}')

mobilitypy version:1.0.3


In [4]:
mobilitypy_tutorial_path = Path().absolute()
save_figure_dir = os.path.join(mobilitypy_tutorial_path,'../imgs')
save_file_name = 'mobilities'
savefigure = True
fig_dpi = 75

## 1.2 Set physical constants

In [5]:
alloy_disordered_effect=1
interface_roughness_effect=1
ion_impurity_effect=1
dislocation_effect=1
deformation_potential_effect=1 
piezoelectric_effect=1
acoustic_phonon_effect=1
polar_optical_phonon_effect=1
total_mobility=1
mobility_model='v2'
#===============================
density_2deg = 10 # 10^12 cm^-2
irf_rms_roughness = 0.3 # nm
irf_corr_length = 3.0 # nm
dislocation_density = 1e2 # 10^8 cm^-2
occup_dislocation = 0.3
n_ion_impurity = 1e2 # 10^14 cm^-3
T=300 #K

# 2. Calculate mobilities

In [6]:
compositions = np.linspace(0,1,101)
mu2deg = Mobility2DCarrier(compositions=compositions, binaries=['AlN', 'GaN'],  
                           pseudomorphic_strain=True, substrate='AlN', alloy_crystal_structure='WZ')

In [7]:
mobility_df = mu2deg.calculate_sheet_mobility(n_2d=density_2deg, rms_roughness=irf_rms_roughness, 
                                              corr_len=irf_corr_length, n_ion_impurity=
                                              n_ion_impurity, n_dis=dislocation_density, 
                                              f_dis=occup_dislocation, T=T,
                                              alloy_disordered_effect=alloy_disordered_effect,
                                              interface_roughness_effect=interface_roughness_effect,
                                              ion_impurity_effect=ion_impurity_effect,
                                              dislocation_effect=dislocation_effect,
                                              deformation_potential_effect=deformation_potential_effect,
                                              piezoelectric_effect=piezoelectric_effect,
                                              acoustic_phonon_effect=acoustic_phonon_effect,
                                              polar_optical_phonon_effect=polar_optical_phonon_effect,
                                              total_mobility=total_mobility,
                                              mobility_model=mobility_model)

In [8]:
mobility_df

,comp,AD,IFR,ION_IMP,DIS_Chg,DIS_Strain,POP,AP,DP,PE,TOT
0,0.000,NaN,32508.978758,9.122365e+06,164513.618785,1.415158e+06,2296.512118,7334.631286,20707.398360,11357.494751,1640.856103
1,0.010,23044.020977,32172.950696,9.029579e+06,163480.526900,1.391169e+06,2275.770373,7219.467403,20428.309141,11165.362933,1516.645790
2,0.020,10940.445178,31843.498169,8.938393e+06,162459.896831,1.367742e+06,2255.374933,7106.616341,20154.432303,10977.302129,1400.746363
3,0.030,6938.190004,31520.443454,8.848768e+06,161451.485689,1.344859e+06,2235.317381,6996.014400,19885.624184,10793.198202,1293.313159
4,0.040,4958.727740,31203.614910,8.760668e+06,160455.057095,1.322505e+06,2215.589564,6887.600097,19621.746132,10612.941000,1194.243248
...,...,...,...,...,...,...,...,...,...,...,...
96,0.960,567.030328,16242.603214,4.324440e+06,100768.440649,3.960796e+05,1193.121560,1847.635111,6461.739373,2587.487379,310.830231
97,0.970,746.695236,16161.321332,4.298557e+06,100338.995039,3.919650e+05,1186.828647,1820.777987,6381.562564,2547.677586,356.422617
98,0.980,1106.785236,16080.949891,4.272941e+06,99912.562529,3.879095e+05,1180.593463,1794.227693,6302.074207,2508.372019,419.657033
99,0.990,2188.533813,16001.474684,4.247588e+06,99489.109269,3.839123e+05,1174.415184,1767.979289,6223.263091,2469.562153,512.973666


In [9]:
composition_axis = np.array(mobility_df['comp'], dtype=float)
mu_total = np.array(mobility_df['TOT'], dtype=float)

In [10]:
# Unit of sheet resitance: ohm per square
sheet_resistance = mu2deg.calculate_sheet_resitance(density_2deg, mu_total)
data_sheet_resitance_2_plot = np.column_stack((composition_axis, sheet_resistance))

In [11]:
# Unit of lateral_FOM: MW/cm^2
lateral_FOM = mu2deg.calculate_figure_of_merit(density_2deg, mu_total)
data_FOM_2_plot = np.column_stack((composition_axis, lateral_FOM))# LFOM: MW/cm^2

# 3. Plot mobilities

In [12]:
plt2deg = Plottings(save_figure_dir=save_figure_dir)
x_label = 'Al composition'
y_label = r'Electron mobility ($\mathrm{cm}^2\mathrm{V}^{-1}\mathrm{s}^{-1}$)'

In [13]:
save_file_name_ = f'{save_file_name}_{T}K.png'
print(f'Save file name: {save_file_name_}')
fig, ax,_ = plt2deg.plot_2d_carrier_mobilities(mobility_df, save_file_name=save_file_name_,
                                               ymin=None, ymax=None, xmax=1, xmin=0, y_scale_log=True, 
                                               annotate_pos=(11,11), show_right_ticks=True,
                                               mode='2d_carrier_mobility', yaxis_label=y_label, 
                                               xaxis_label=x_label, color=None, color_map='viridis', 
                                               savefig=savefigure, dpi=fig_dpi, show_plot=True)

Save file name: mobilities_300K.png


In [14]:
save_file_name_ = f'LFOM_{T}K.png'
fig, ax, _ = plt2deg.plot_2d(data_FOM_2_plot, save_file_name=save_file_name_,
                             ymin=None, ymax=None, xmax=1.0, xmin=0, y_scale_log=True,
                             yaxis_label=r'LFOM (MW/cm$^2$)', xaxis_label=x_label,
                             color='k', color_map='viridis', savefig=savefigure, 
                             dpi=fig_dpi, show_plot=False)
ax.axhline(y=data_FOM_2_plot[0,1], c='k', ls='--')
#plt.show()

In [15]:
save_file_name_ = f'LFOM_norm_{T}K.png'
YY = data_FOM_2_plot.copy()
YY[:, 1] /= YY[0, 1]
fig, ax, _ = plt2deg.plot_2d(YY, save_file_name=save_file_name_,
                             ymin=None, ymax=None, xmax=1, xmin=0, y_scale_log=True,
                             yaxis_label=r'LFOM$_\mathrm{AlGaN}$ / LFOM$_\mathrm{GaN}$', 
                             xaxis_label=x_label, color='k', color_map='viridis', 
                             savefig=0, show_plot=False)
ax.axhline(y=1, c='k', ls='--')
plt2deg.save_figure(save_file_name_,fig=fig, savefig=savefigure, dpi=fig_dpi)

In [16]:
save_file_name_ = f'sheet_resistamce_{T}K.png'
#data_sheet_resitance_2_plot[:, 1] /= data_sheet_resitance_2_plot[0, 1]
fig, ax, _ = plt2deg.plot_2d(data_sheet_resitance_2_plot, save_file_name=save_file_name_,
                             ymin=None, ymax=None, xmax=1, xmin=0, y_scale_log=True,
                             yaxis_label=r'Sheet resistance ($\Omega /\boxdot$)', 
                             xaxis_label=x_label, color='k', color_map='viridis', 
                             savefig=0, show_plot=False)
#ax.axhline(y=data_FOM_2_plot[0,1], c='k', ls='--')
plt2deg.save_figure(save_file_name_,fig=fig, savefig=savefigure, dpi=fig_dpi)